# Maternal Assertion Verification and Evidence Network (MAVEN)

## Maternal Health Misinformation Detection Pipeline

Ingests any body of text and outputs per-chunk "misinformation scores" using
[PubMedBERT embeddings](https://huggingface.co/NeuML/pubmedbert-base-embeddings), a biomedical sentence encoder fine-tuned on PubMed abstracts.

### Architecture Flow

```
Raw text
    │
    ▼
[1] Text Segmentation. Chunks (sentence / paragraph / sliding window)
    │
    ▼
[2] PubMedBERT Embeddings. 768-dim L2-normalized vector per chunk
    │
    ▼
[3] Marker Computation ─┬─ authority_sim    cosine sim to vetted anchor centroid
                        ├─ misinfo_sim      cosine sim to misinformation anchor centroid
                        ├─ claim_delta      authority_sim − misinfo_sim
                        └─ isolation_score  Isolation Forest anomaly vs. authoritative corpus
    │
    ▼
[4] Composite Score; weighted misinfo_score within [0, 1] per chunk → flagged boolean
    │
    ▼
Output DataFrame
```

### Scale handling

| Text size | Strategy |
|---|---|
| < 300 tokens | Sentence-level chunking |
| 300–3 000 tokens | Paragraph-level chunking |
| > 3 000 tokens | Sliding-window chunking (configurable window + stride) |
| Corpus / batch | Batched encoding via `sentence-transformers` |

In [1]:
!pip install -q sentence-transformers scikit-learn nltk pandas numpy torch

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

## 1. Text Segmentation

Splits a raw text body into semantically coherent chunks before embedding.

| Mode | Granularity | When to use |
|---|---|---|
| `sentence` | Finest | Short passages; high-precision flagging |
| `paragraph` | Default | Article-length text |
| `sliding_window` | Fixed token windows + overlap | Long documents; prevents context loss at boundaries |

`chunk_text(text, mode='auto')` dispatches automatically based on approximate token count.
The stride in sliding-window mode controls how much adjacent chunks overlap. Larger overlap preserves more cross-boundary context at the cost of redundant scoring.

In [ ]:
import re
from typing import List, Tuple
from nltk.tokenize import sent_tokenize

# Thresholds for auto-dispatch 
SENTENCE_THRESHOLD  = 300    # tokens: below → sentence mode
PARAGRAPH_THRESHOLD = 3_000  # tokens: below → paragraph mode; above → sliding window


def _approx_tokens(text: str) -> int:
    return len(text.split())


def _by_sentence(text: str) -> List[str]:
    return [s.strip() for s in sent_tokenize(text) if len(s.strip()) > 20]


def _by_paragraph(text: str) -> List[str]:
    paras = [p.strip() for p in re.split(r'\n{2,}', text)]
    return [p for p in paras if len(p) > 40]


def _by_sliding_window(
    text: str,
    window: int = 200, # words per chunk
    stride: int = 100, # words between chunk starts (50 % overlap by default)
) -> List[str]:
    words = text.split()
    chunks = []
    for i in range(0, len(words), stride):
        chunk = ' '.join(words[i : i + window])
        if len(chunk) > 40:
            chunks.append(chunk)
        if i + window >= len(words):
            break
    return chunks


def chunk_text(
    text: str,
    mode: str = 'auto', # 'auto' | 'sentence' | 'paragraph' | 'sliding_window'
    window: int = 200,
    stride: int = 100,
) -> Tuple[List[str], str]:
    """Return (chunks, mode_used)."""
    if mode == 'auto':
        n = _approx_tokens(text)
        if n < SENTENCE_THRESHOLD:
            mode = 'sentence'
        elif n < PARAGRAPH_THRESHOLD:
            mode = 'paragraph'
        else:
            mode = 'sliding_window'

    dispatch = {
        'sentence':       lambda: _by_sentence(text),
        'paragraph':      lambda: _by_paragraph(text),
        'sliding_window': lambda: _by_sliding_window(text, window, stride),
    }
    return dispatch[mode](), mode


# sanity check 
_sample = (
    'Prenatal folic acid reduces neural tube defect risk.\n\n'
    'Vaccines during pregnancy are safe and recommended by the CDC and WHO.\n\n'
    'Misinformation claims that ultrasounds harm fetal brain development.'
)
for _mode in ('sentence', 'paragraph', 'sliding_window'):
    _chunks, _used = chunk_text(_sample, mode=_mode)
    print(f'{_used:>16} → {len(_chunks)} chunk(s)')

        sentence → 3 chunk(s)
       paragraph → 3 chunk(s)
  sliding_window → 1 chunk(s)


## 2. PubMedBERT Embeddings

**Model:** [`NeuML/pubmedbert-base-embeddings`](https://huggingface.co/NeuML/pubmedbert-base-embeddings)
A `sentence-transformers` model fine-tuned on PubMed biomedical text pairs.
Produces **768-dimensional, L2-normalized** vectors.

**Motivation Behind PubMedBERT over general BERT:**
General-domain encoders conflate clinical terminology with colloquial usage.
PubMedBERT captures *biomedical* semantic distance, making cosine similarity
meaningful and more explanatory for distinguishing evidence-based claims from unsupported ones.

**Key properties used here:**
- Because embeddings are L2-normalized, `cosine_similarity(a, b) = dot(a, b)` — fast and numerically stable.
- Batch encoding amortizes model overhead; scale from single sentences to millions of passages by adjusting `batch_size`.

In [ ]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

print('Loading PubMedBERT embeddings model...')
embed_model = SentenceTransformer('NeuML/pubmedbert-base-embeddings', device=DEVICE)
EMBED_DIM = 768
print('Model loaded.')


def embed(texts: List[str], batch_size: int = 32, show_progress: bool = False) -> np.ndarray:
    """
    Encode texts into L2-normalized 768-dim PubMedBERT embeddings.
    Returns ndarray of shape (len(texts), 768).
    Handles any corpus size via batching.
    """
    if not texts:
        return np.empty((0, EMBED_DIM))
    return embed_model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,   # L2-normalize → cosine sim = dot product
        show_progress_bar=show_progress,
        convert_to_numpy=True,
    )


# Smoke test 
_test = embed(['Prenatal care reduces maternal mortality.'])
print(f'Embedding shape: {_test.shape}  |  L2-norm: {np.linalg.norm(_test[0]):.4f}')

c:\Users\Thomas\Desktop\projects\hmd-dept-of-mch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu
Loading PubMedBERT embeddings model...


c:\Users\Thomas\Desktop\projects\hmd-dept-of-mch\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Thomas\.cache\huggingface\hub\models--NeuML--pubmedbert-base-embeddings. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4581.93it/s]


Model loaded.
Embedding shape: (1, 768)  |  L2-norm: 1.0000


## 3. Quantifiable Misinformation Markers

Four continuous markers are computed per chunk from its PubMedBERT embedding.

| Marker | Range | High value means |
|---|---|---|
| `authority_sim` | [−1, 1] | Chunk is semantically close to verified, authoritative claims |
| `misinfo_sim` | [−1, 1] | Chunk is semantically close to known misinformation claims |
| `claim_delta` | [−2, 2] | `authority_sim − misinfo_sim`; positive -> leans authoritative, negative -> leans misinformation |
| `isolation_score` | [0, 1] | Isolation Forest anomaly score vs. authoritative corpus; high -> anomalous relative to vetted content |

**Anchor sets** are the critical domain input Small curated sets of sentence-level claims:
one for verified maternal health facts, one for known misinformation.
The anchor centroids are compared against each chunk via cosine similarity.

> **Action required (M1):** Replace placeholder anchors with domain-validated
> claims reviewed by Dr. Bazzano and the research team. More anchors improve centroid stability;
> aim for >= 20 per set before final evaluation.

In [ ]:
from sklearn.ensemble import IsolationForest

# Anchor sets; REPLACE with domain-validated claims before Milestone 1 
AUTHORITY_ANCHORS: List[str] = [
    'Prenatal folic acid supplementation significantly reduces the risk of neural tube defects.',
    'The CDC and WHO recommend influenza vaccination during any trimester of pregnancy.',
    'Gestational diabetes is managed through diet, exercise, and medication when necessary.',
    'Breastfeeding provides immune protection to infants through maternal antibodies.',
    'Regular prenatal care is associated with reduced maternal and infant mortality.',
    'Preeclampsia is characterized by high blood pressure and requires clinical management.',
    'Group B Streptococcus screening is routinely performed at 36 weeks of gestation.',
    'Maternal pertussis vaccination protects newborns before they can be immunized.',
    'Low-dose aspirin may be recommended to reduce preeclampsia risk in high-risk pregnancies.',
    'Postpartum depression affects approximately 1 in 8 women and is treatable.',
]

MISINFO_ANCHORS: List[str] = [
    'Vaccines during pregnancy cause miscarriage and permanent fetal harm.',
    'Epidurals damage the spine and impair fetal brain development.',
    'Folic acid supplements are unnecessary and potentially toxic to the fetus.',
    'Ultrasounds during pregnancy are dangerous and should be avoided.',
    'C-sections cause long-term immune deficiencies in children.',
    'Herbal teas during pregnancy can replace prenatal vitamins entirely.',
    'Breastfeeding mothers must avoid all medications, including prescribed antibiotics.',
    'Induced labor always leads to worse outcomes than spontaneous labor.',
    'Gestational diabetes can be cured by eating only organic foods.',
    'Natural childbirth without any medical intervention is always the safest option.',
]

# Embed anchors and build centroids 
print('Embedding authority anchors...')
authority_embs = embed(AUTHORITY_ANCHORS)          # (n_auth, 768)
print('Embedding misinformation anchors...')
misinfo_embs   = embed(MISINFO_ANCHORS)            # (n_mis, 768)

# L2-normalize centroids so cosine sim = dot product holds
def _l2(v: np.ndarray) -> np.ndarray:
    return v / (np.linalg.norm(v) + 1e-10)

authority_centroid = _l2(authority_embs.mean(axis=0))[np.newaxis, :]  # (1, 768)
misinfo_centroid   = _l2(misinfo_embs.mean(axis=0))[np.newaxis, :]    # (1, 768)

# Isolation Forest coef: trained on authoritative embeddings 
# contamination='auto': assumes authoritative anchors are clean inliers.
# Scores how anomalous a chunk is relative to vetted biomedical content.
iso_forest = IsolationForest(n_estimators=200, contamination='auto', random_state=42)
iso_forest.fit(authority_embs)
print('Anchors embedded | Isolation Forest fitted.')


def compute_markers(chunk_embs: np.ndarray) -> dict:
    """
    Compute four scalar markers for each of N chunk embeddings.
    Args:
        chunk_embs: (N, 768) L2-normalized embeddings
    Returns:
        dict of arrays each of length N
    """
    # Cosine similarities (dot product; embeddings are L2-normalized)
    auth_sim    = (chunk_embs @ authority_centroid.T).flatten()  # (N,)
    misinfo_sim = (chunk_embs @ misinfo_centroid.T).flatten()    # (N,)
    claim_delta = auth_sim - misinfo_sim                         # (N,) in [-2, 2] # Isolation Forest: decision_function is lower for anomalies.
    # Negate and min-max scale to [0, 1] so higher = more anomalous.
    iso_raw = -iso_forest.decision_function(chunk_embs)          # (N,)
    lo, hi  = iso_raw.min(), iso_raw.max()
    iso_score = (iso_raw - lo) / (hi - lo + 1e-10)              # (N,) in [0, 1]

    return {
        'authority_sim':   auth_sim,
        'misinfo_sim':     misinfo_sim,
        'claim_delta':     claim_delta,
        'isolation_score': iso_score,
    }

Embedding authority anchors...
Embedding misinformation anchors...
Anchors embedded | Isolation Forest fitted.


## 4. Composite Scoring & Output

The four markers are combined into a single **`misinfo_score ∈ [0, 1]`** per chunk.

### Formula

```
claim_delta_score = (1 − clip(claim_delta, −1, 1)) / 2     # maps [−1, 1] → [1, 0]
misinfo_score     = 0.70 × claim_delta_score + 0.30 × isolation_score
```

**Weight rationale:**
- `claim_delta` (weight 0.70) is the primary signal. It directly contrasts semantic proximity
  to authoritative vs. misinformation content.
- `isolation_score` (weight 0.30) supplements by detecting claims semantically distant from
  *any* authoritative content, catching novel misinformation outside the anchor vocabulary.

Raw `authority_sim` and `misinfo_sim` are preserved in the output DataFrame for transparency
and downstream analysis but are not included in the composite to avoid double-counting.

**Threshold:** A chunk is flagged when `misinfo_score ≥ FLAG_THRESHOLD` (default 0.60).
TODO:
Tune this threshold against labeled data at (M2).

In [7]:
import pandas as pd

FLAG_THRESHOLD = 0.60  # tune against labeled data at Milestone 2


def score_text(
    text: str,
    chunk_mode: str = 'auto',
    window: int = 200,
    stride: int = 100,
    batch_size: int = 32,
    flag_threshold: float = FLAG_THRESHOLD,
) -> pd.DataFrame:
    """
    Intake any body of text; return a DataFrame of per-chunk misinformation scores.

    Columns:
        chunk           raw text segment
        chunk_mode      segmentation strategy applied
        authority_sim   cosine sim to authoritative anchor centroid  (higher = better)
        misinfo_sim     cosine sim to misinformation anchor centroid  (higher = worse)
        claim_delta     authority_sim − misinfo_sim                  (higher = better)
        isolation_score anomaly score vs. authoritative corpus       (higher = worse)
        misinfo_score   composite misinformation score ∈ [0, 1]     (higher = worse)
        flagged         True when misinfo_score ≥ flag_threshold
    """
    chunks, mode_used = chunk_text(text, mode=chunk_mode, window=window, stride=stride)
    if not chunks:
        return pd.DataFrame()

    chunk_embs = embed(chunks, batch_size=batch_size)
    markers    = compute_markers(chunk_embs)

    # Normalize claim_delta from [-1, 1] (clipped) → [0, 1]
    # High delta (authoritative) → low score; low/negative delta → high score
    delta_clipped = np.clip(markers['claim_delta'], -1.0, 1.0)
    delta_score   = (1.0 - delta_clipped) / 2.0           # [0, 1]

    misinfo_score = 0.70 * delta_score + 0.30 * markers['isolation_score']

    return pd.DataFrame({
        'chunk':           chunks,
        'chunk_mode':      mode_used,
        'authority_sim':   markers['authority_sim'].round(4),
        'misinfo_sim':     markers['misinfo_sim'].round(4),
        'claim_delta':     markers['claim_delta'].round(4),
        'isolation_score': markers['isolation_score'].round(4),
        'misinfo_score':   misinfo_score.round(4),
        'flagged':         misinfo_score >= flag_threshold,
    })


# Demo 
DEMO_TEXT = """
Adequate prenatal care is associated with significantly reduced rates of maternal
and infant mortality across all demographic groups.

Folic acid supplementation before and during early pregnancy prevents the majority
of neural tube defects, according to CDC guidelines.

Vaccines administered during pregnancy are known to cause permanent damage to the
developing fetal brain. Many doctors secretly know this but continue vaccinating
for financial incentives.

Preeclampsia is managed through blood pressure monitoring and, when necessary,
early delivery. It affects 5-8% of pregnancies worldwide.

Ultrasounds are secretly harmful to fetal development and should be avoided.
The medical establishment suppresses this information.
"""

df_scores = score_text(DEMO_TEXT)

print(f'Chunks scored : {len(df_scores)}')
print(f'Flagged       : {df_scores["flagged"].sum()}')
print(f'Mean score    : {df_scores["misinfo_score"].mean():.3f}')
display(df_scores)

print('\nFlagged chunks')
display(
    df_scores[df_scores['flagged']]
    [['chunk', 'misinfo_score', 'claim_delta', 'isolation_score']]
    .reset_index(drop=True)
)

Chunks scored : 8
Flagged       : 1
Mean score    : 0.509


,chunk,chunk_mode,authority_sim,misinfo_sim,claim_delta,isolation_score,misinfo_score,flagged
0,Adequate prenatal care is associated with sign...,sentence,0.5793,0.4112,0.1681,0.6657,0.4909,False
1,Folic acid supplementation before and during e...,sentence,0.6061,0.5152,0.0909,0.0000,0.3182,False
2,Vaccines administered during pregnancy are kno...,sentence,0.5017,0.6867,-0.1850,0.5087,0.5674,False
3,Many doctors secretly know this but continue v...,sentence,0.2468,0.3698,-0.1230,0.5852,0.5686,False
4,Preeclampsia is managed through blood pressure...,sentence,0.6337,0.4713,0.1624,0.3590,0.4009,False
5,It affects 5-8% of pregnancies worldwide.,sentence,0.5105,0.5581,-0.0477,0.2918,0.4542,False
6,Ultrasounds are secretly harmful to fetal deve...,sentence,0.5072,0.7638,-0.2565,0.4608,0.5780,False
7,The medical establishment suppresses this info...,sentence,0.1189,0.2412,-0.1222,1.0000,0.6928,True



Flagged chunks


,chunk,misinfo_score,claim_delta,isolation_score
0,The medical establishment suppresses this info...,0.6928,-0.1222,1.0
